# Evaluating GraphRAG with RAGAS



**Course module:** Module 8  

**Audience:** Beginners who completed (or are running alongside) `Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`.



## Course description



Evaluate the output of your **GraphRAG** application built on **Neo4j** and compare it with a **vector-only RAG** baseline



You will learn how to:



1. Wire **Neo4j vector RAG** and **GraphRAG** answer pipelines for evaluation.

2. Build a **ground-truth question set** for the maritime lab graph.

3. Measure **accuracy** with **RAGAS** (`faithfulness`, `answer_relevancy`).

4. Compare **latency** and **token cost** between RAG and GraphRAG.

5. Inspect low-score samples and iterate on retrieval settings.



> **Language:** All instructional text is in **English**.



> **Setup (required before code):** Complete **`NEO4J_SETUP.md`** and **`LLM_MODEL_SETUP.md`**.  

> For local LLMs, run `ollama serve` and use **`ollama_model_runner.py`** as described in the LLM guide.



### How to use this notebook



1. Read each **markdown** cell before running the **code** cell below it.

2. Run cells **in order** from Step 0 downward.

3. **Prerequisite:** The LangChain lab graph must exist in Neo4j (Sections 3.1.3 and 3.2 of the companion notebook). Step 1 verifies this.

4. Re-running is safe: evaluation outputs go to `Module_8/outputs/eval_graphrag/`.



## Prerequisites



Before taking this course, you should have:



- Completed (or be ready to run) **`Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`** — especially:

  - Seeded `LangChainLab` nodes (ports, Maersk, Panama Canal, chunks).

  - Built the Neo4j vector index `langchain_lab_chunk_index`.

- Basic understanding of **RAG**, **GraphRAG**, and **evaluation metrics**.

- Python, Jupyter, and a running **Neo4j 5.15+** instance.



### Concepts you will use



| Concept | Meaning in this course |

|---------|------------------------|

| **Vector RAG** | Retrieve top-k text chunks by embedding similarity, then generate an answer |

| **GraphRAG** | Vector retrieval + graph expansion (`MENTIONS`, `USES_ROUTE`, …) before generation |

| **Ground truth** | Reference answer for each evaluation question |

| **Faithfulness** | RAGAS metric: is the answer supported by retrieved context? |

| **Answer relevancy** | RAGAS metric: does the answer address the question? |

| **Latency** | Wall-clock time per question (retrieval + generation) |

| **Cost proxy** | Token count from Ollama (`eval_count`) or API usage for hosted models |



## Course outline



| Step | Topic |

|------|--------|

| **0** | Environment, Neo4j, LLM, RAGAS dependencies |

| **1** | Verify lab graph and reconnect vector index |

| **2** | Build vector RAG and GraphRAG answer functions |

| **3** | Ground-truth dataset for the maritime lab |

| **4** | Run both pipelines and collect timing / token stats |

| **5** | Score with RAGAS |

| **6** | Compare time, cost, and accuracy |

| **7** | Inspect failures and iterate |

| **8** | Save artifacts and checklist |



---



# Step 0 — Development environment



This section prepares Python packages, loads `.env`, and confirms **Neo4j** and your **LLM** are reachable.



### Before you run any code



1. Complete **`NEO4J_SETUP.md`** — Aura, Desktop, or Docker; note URI, username, password.

2. Complete **`LLM_MODEL_SETUP.md`** — Ollama + `ollama_model_runner.py` (recommended) or OpenAI.

3. Copy `Module_8/.env.example` → `Module_8/.env` and fill in credentials.

4. Start Neo4j and (for Ollama) run `ollama serve` in a terminal.

5. Run **`Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`** through Section **3.2.3** so the lab graph and vector index exist.



### Step 0a — Install Python packages



We add **RAGAS** and **datasets** for evaluation, plus the same Neo4j / LangChain stack as the companion notebook.



| Package | Role |

|---------|------|

| `ragas` | Faithfulness, answer relevancy, and other RAG metrics |

| `datasets` | Hugging Face `Dataset` format for RAGAS |

| `langchain-neo4j` | `Neo4jGraph`, `Neo4jVector` |

| `pandas`, `tqdm` | Tables and progress bars during batch evaluation |



**Version note (important):**

- Keep **`langchain-community>=0.4.2`** — required by `langchain-experimental` and other Module 8 notebooks.
- **Do not downgrade** `langchain-community` to fix RAGAS (that causes the pip conflict you may have seen).
- `ragas` 0.4.x imports a removed VertexAI path; this notebook applies a **shim in Step 5a.0** before importing RAGAS.
- You do **not** need `langchain-google-vertexai` for the Ollama lab.



In [11]:
# Step 0a — Install dependencies (run once per environment)

%pip install -q -U "langchain-community>=0.4.2,<1.0.0" \
    neo4j python-dotenv requests langchain langchain-classic langchain-neo4j \
    langchain-openai langchain-huggingface sentence-transformers \
    ragas datasets pandas tqdm




[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Expected output (Step 0a):** `Note: you may need to restart the kernel...` with **no red ERROR lines**.



If you still see `langchain-community 0.3.x` conflicts, re-run this cell once (it upgrades `langchain-community` to `>=0.4.2`), then **restart the kernel** and continue from Step 0b.



**Note:** Run this cell once per virtual environment. If versions conflict, restart the kernel after install.



**Expected output:** Package installation messages (or `Requirement already satisfied`). No errors.



### Step 0b.1 — Resolve the `Module_8` directory and load `.env`



Jupyter's working directory depends on how you launched the notebook:



- From repo root → we detect `Module_8/` as a subfolder.

- From `Module_8/` → we use the current folder.



We then call `load_dotenv(MODULE_DIR / '.env')` so `NEO4J_*` and `LLM_*` variables are available to every later cell without hard-coding secrets in the notebook.



We also create **`outputs/eval_graphrag/`** early so every save step has a known folder.



In [12]:
# Step 0b.1 — Module path and .env

import os

from pathlib import Path

from dotenv import load_dotenv



MODULE_DIR = Path('.').resolve()

if MODULE_DIR.name != 'Module_8':

    candidate = MODULE_DIR / 'Module_8'

    if candidate.is_dir():

        MODULE_DIR = candidate.resolve()

load_dotenv(MODULE_DIR / '.env')

OUTPUT_DIR = MODULE_DIR / 'outputs' / 'eval_graphrag'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Module directory:', MODULE_DIR)

print('Evaluation outputs:', OUTPUT_DIR)



Module directory: /home/ethan/newgen/KMOU_Course/Module_8
Evaluation outputs: /home/ethan/newgen/KMOU_Course/Module_8/outputs/eval_graphrag


**Expected output:** A path ending in `.../Module_8` and an evaluation output directory under it.



If the path is wrong, use **File → Open** from inside `Module_8/` or set the Jupyter root accordingly.



### Step 0b.2 — Neo4j connection settings



These variables must match your deployment (see **`NEO4J_SETUP.md`**):



| Variable | Typical local (Docker/Desktop) | Aura cloud |

|----------|-------------------------------|------------|

| `NEO4J_URI` | `neo4j://localhost:7687` | `neo4j+s://....databases.neo4j.io` |

| `NEO4J_USERNAME` | `neo4j` | `neo4j` |

| `NEO4J_PASSWORD` | your instance password | from Aura credentials file |

| `NEO4J_DATABASE` | `neo4j` | `neo4j` |



**Security:** Never commit real passwords to Git. Use `.env` only (gitignored).



**Troubleshooting:**



- `NEO4J_PASSWORD is empty` → copy `Module_8/.env.example` to `.env` and fill values.

- Docker users: password in `.env` must match `NEO4J_AUTH` (see **`NEO4J_SETUP.md`**, Option C).



In [13]:
# Step 0b.2 — Neo4j settings

NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7687')

NEO4J_USERNAME = os.getenv('NEO4J_USERNAME', 'neo4j')

NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')

NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

print('Neo4j URI:', NEO4J_URI)

print('Neo4j database:', NEO4J_DATABASE)



Neo4j URI: neo4j://localhost:7687
Neo4j database: neo4j


### Step 0b.3 — LLM backend and evaluation knobs



This notebook uses an LLM in **three** places:



| Use case | Object | Why |

|----------|--------|-----|

| RAG / GraphRAG answers | `chat_llm` or `call_ollama_runner()` | Generate final answers under test |

| RAGAS scoring | `ragas_llm` | Judge faithfulness and relevancy |

| Vector search | `embeddings` | Same model as the LangChain lab index |



| `LLM_BACKEND` | How the notebook calls the model |

|---------------|-----------------------------------|

| `ollama` (default) | Subprocess → `ollama_model_runner.py` → Ollama HTTP API |

| `openai` | `ChatOpenAI` with `OPENAI_API_KEY` |



**Evaluation-specific setting:**



- **`EVAL_RETRIEVAL_K`** (default `3`) — how many chunks each retriever returns. Higher `k` → more context, longer prompts, slower runs. You will compare pipelines at the **same** `k` for a fair benchmark.



In [14]:
# Step 0b.3 — LLM settings

LLM_BACKEND = os.getenv('LLM_BACKEND', 'ollama').lower()

OLLAMA_HOST = os.getenv('OLLAMA_HOST', 'http://localhost:11434')

OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'llama3.2:3b')

OLLAMA_TEMPERATURE = float(os.getenv('OLLAMA_TEMPERATURE', '0'))

OLLAMA_MAX_TOKENS = int(os.getenv('OLLAMA_MAX_TOKENS', '2048'))

OPENAI_MODEL = os.getenv('OPENAI_MODEL', 'gpt-4o-mini')

RETRIEVAL_K = int(os.getenv('EVAL_RETRIEVAL_K', '3'))

print('LLM backend:', LLM_BACKEND)

print('Retrieval k:', RETRIEVAL_K)

if LLM_BACKEND == 'ollama':

    print('Ollama model:', OLLAMA_MODEL, '@', OLLAMA_HOST)



LLM backend: ollama
Retrieval k: 3
Ollama model: llama3.1:8b @ http://localhost:11434


### Step 0c — Verify Neo4j connectivity



We open a short-lived **Bolt** session using the official `neo4j` driver — the same protocol LangChain's `Neo4jGraph` uses under the hood.



**If this cell fails:**



| Error | Fix |

|-------|-----|

| `NEO4J_PASSWORD is empty` | Fill `Module_8/.env` |

| `ServiceUnavailable` | Database not running or wrong URI scheme |

| `Authentication failed` | Password mismatch (Docker `NEO4J_AUTH` must match `.env`) |



In [15]:
# Step 0c — Neo4j smoke test

from neo4j import GraphDatabase



if not NEO4J_PASSWORD:

    raise ValueError('NEO4J_PASSWORD is empty. See NEO4J_SETUP.md and Module_8/.env')



_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with _driver.session(database=NEO4J_DATABASE) as session:

    msg = session.run('RETURN "Neo4j connection OK" AS message').single()['message']

    print(msg)

_driver.close()

print('Connectivity check passed.')



Neo4j connection OK
Connectivity check passed.


**Expected output:** `Neo4j connection OK` then `Connectivity check passed.`



You can also confirm in **Neo4j Browser** (`http://localhost:7474` for Desktop/Docker):



```cypher

RETURN "Neo4j connection OK" AS message;

```



### Step 0d — Ollama runner helpers



LangChain expects model objects (`LLM`, `BaseChatModel`). Our course runner is a **CLI script** that returns JSON on stdout. We bridge the gap with:



1. **`call_ollama_runner()`** — writes prompt to temp file, runs subprocess, parses JSON.

2. **`OllamaRunnerLLM`** — classic LLM interface (not used heavily here, kept for consistency).

3. **`OllamaRunnerChatModel`** — used by RAG-style chains that expect chat messages.



This evaluation notebook **extends** the runner wrapper to capture operational metrics:



| Field | Meaning |

|-------|---------|

| `latency_sec` | Wall-clock time for the subprocess call |

| `eval_count` | Ollama output tokens — **cost proxy** for local models |



See **`LLM_MODEL_SETUP.md`** for installing Ollama, pulling models, and running the runner from a terminal.



#### Step 0d.1 — Imports and locate `ollama_model_runner.py`



Search order: `Module_8/` → `Module_4/` (fallback) → current directory.



We import **`pandas`** and **`tqdm`** here because later steps build evaluation tables and show progress bars during batch runs.



In [16]:
# Step 0d.1 — Imports and locate ollama_model_runner.py

import json

import subprocess

import sys

import tempfile

import time

from typing import Any, List, Optional



import pandas as pd

from tqdm.auto import tqdm



from langchain_core.callbacks import CallbackManagerForLLMRun

from langchain_core.language_models.chat_models import BaseChatModel

from langchain_core.language_models.llms import LLM

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage

from langchain_core.outputs import ChatGeneration, ChatResult



pd.set_option('display.max_colwidth', 120)





def _resolve_ollama_runner_path() -> Path:

    for candidate in (

        MODULE_DIR / 'ollama_model_runner.py',

        MODULE_DIR.parent / 'Module_4' / 'ollama_model_runner.py',

        Path('ollama_model_runner.py'),

    ):

        if candidate.exists():

            return candidate.resolve()

    raise FileNotFoundError('ollama_model_runner.py not found (see LLM_MODEL_SETUP.md)')





OLLAMA_RUNNER = _resolve_ollama_runner_path()

print('OLLAMA_RUNNER:', OLLAMA_RUNNER)



OLLAMA_RUNNER: /home/ethan/newgen/KMOU_Course/Module_8/ollama_model_runner.py


#### Step 0d.2 — `call_ollama_runner()` with timing and token stats



Parameters passed to the script:



| Flag | Source |

|------|--------|

| `--host` | `OLLAMA_HOST` |

| `--models` | `OLLAMA_MODEL` |

| `--prompt-file` | temp file with full prompt |

| `--temperature` | `OLLAMA_TEMPERATURE` (0 for deterministic lab) |

| `--max-tokens` | `OLLAMA_MAX_TOKENS` (raise if answers are cut off) |



Unlike the LangChain lab notebook, this function returns a **dictionary** (not just a string) so Steps 4 and 6 can aggregate latency and token usage per question.



**Why subprocess?** Keeps the Jupyter kernel light; long prompts run in a separate process (same pattern as other KMOU modules).



In [17]:
# Step 0d.2 — call_ollama_runner() with timing and token stats

def call_ollama_runner(prompt: str, *, model: str | None = None) -> dict:

    """Return response text plus latency and eval_count from Ollama JSON."""

    model_arg = model or OLLAMA_MODEL

    with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as pf:

        path = pf.name

        pf.write(prompt)

    t0 = time.perf_counter()

    try:

        cmd = [

            sys.executable, str(OLLAMA_RUNNER),

            '--host', OLLAMA_HOST,

            '--models', model_arg,

            '--prompt-file', path,

            '--temperature', str(OLLAMA_TEMPERATURE),

            '--max-tokens', str(OLLAMA_MAX_TOKENS),

        ]

        run = subprocess.run(cmd, capture_output=True, text=True)

        latency_sec = time.perf_counter() - t0

        if run.returncode != 0:

            raise RuntimeError(f'runner exit {run.returncode}\n{run.stderr[:4000]}')

        payload = json.loads(run.stdout)

        first = (payload.get('outputs') or [{}])[0]

        if first.get('status') != 'ok':

            raise RuntimeError(f'Ollama error: {first}')

        return {

            'response': (first.get('response') or '').strip(),

            'latency_sec': latency_sec,

            'eval_count': int(first.get('eval_count') or 0),

            'model': model_arg,

        }

    finally:

        Path(path).unlink(missing_ok=True)



#### Step 0d.3 — LangChain adapter classes



- **`OllamaRunnerLLM`** implements `_call(prompt) → str` for classic LLM chains.

- **`OllamaRunnerChatModel`** flattens message lists into one prompt string (sufficient for this course; production apps may prefer native chat APIs).



These adapters are identical to **`Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`** so learners see one consistent pattern across Module 8 notebooks.



In [18]:
# Step 0d.3 — LangChain LLM adapters (same as LangChain lab notebook)

class OllamaRunnerLLM(LLM):

    model: str = OLLAMA_MODEL



    @property

    def _llm_type(self) -> str:

        return 'ollama_runner'



    def _call(self, prompt: str, stop: Optional[List[str]] = None,

              run_manager: Optional[CallbackManagerForLLMRun] = None, **kwargs: Any) -> str:

        return call_ollama_runner(prompt, model=self.model)['response']





class OllamaRunnerChatModel(BaseChatModel):

    model: str = OLLAMA_MODEL



    @property

    def _llm_type(self) -> str:

        return 'ollama_runner_chat'



    def _messages_to_prompt(self, messages: List[BaseMessage]) -> str:

        parts = []

        for m in messages:

            if isinstance(m, SystemMessage):

                parts.append(f'System: {m.content}')

            elif isinstance(m, HumanMessage):

                parts.append(f'User: {m.content}')

            elif isinstance(m, AIMessage):

                parts.append(f'Assistant: {m.content}')

            else:

                parts.append(str(m.content))

        parts.append('Assistant:')

        return '\n'.join(parts)



    def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None,

                  run_manager: Optional[CallbackManagerForLLMRun] = None, **kwargs: Any) -> ChatResult:

        text = call_ollama_runner(self._messages_to_prompt(messages), model=self.model)['response']

        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])



#### Step 0d.4 — Instantiate `chat_llm` and `ragas_llm`



| Variable | Used for |

|----------|----------|

| `chat_llm` | Optional direct OpenAI path; Ollama answers use `call_ollama_runner()` for metrics |

| `ragas_llm` | RAGAS metric computation (makes its own LLM calls per row) |



**Why two LLM objects for Ollama?**



- RAGAS expects an HTTP client (`ChatOpenAI`) pointing at Ollama's **OpenAI-compatible** endpoint (`{OLLAMA_HOST}/v1`).

- Our RAG answer functions use **`call_ollama_runner()`** so we capture `eval_count` and precise latency from the runner JSON.



Using the **same model name** for both keeps evaluation comparable, though RAGAS adds extra LLM calls beyond answer generation.



In [19]:
# Step 0d.4 — Instantiate chat_llm and ragas_llm

if LLM_BACKEND == 'openai':

    if not os.getenv('OPENAI_API_KEY'):

        raise ValueError('OPENAI_API_KEY required when LLM_BACKEND=openai')

    from langchain_openai import ChatOpenAI

    chat_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)

    ragas_llm = chat_llm

    print(f'Using OpenAI: {OPENAI_MODEL}')

elif LLM_BACKEND == 'ollama':

    chat_llm = OllamaRunnerChatModel()

    ragas_llm = __import__('langchain_openai', fromlist=['ChatOpenAI']).ChatOpenAI(

        model=OLLAMA_MODEL,

        api_key=os.getenv('OPENAI_API_KEY', 'ollama'),

        base_url=f"{OLLAMA_HOST.rstrip('/')}/v1",

        temperature=0,

    )

    print(f'Using Ollama runner: {OLLAMA_MODEL} @ {OLLAMA_HOST}')

else:

    raise ValueError("Set LLM_BACKEND to 'ollama' or 'openai'")



Using Ollama runner: llama3.1:8b @ http://localhost:11434


#### Step 0d.5 — Embeddings for RAGAS and Neo4jVector reconnect



We use **`sentence-transformers/all-MiniLM-L6-v2`** (384 dimensions) — the same default as the LangChain lab notebook.



| Consumer | Why embeddings matter |

|----------|----------------------|

| **Neo4jVector** | Must match the model used when the index was built (Step 1c) |

| **RAGAS `answer_relevancy`** | Embeds generated answers and synthetic questions for similarity scoring |



Override with env var `RAG_EMBED_MODEL` only if you rebuilt the vector index with a different model.



**First run:** downloads model weights from Hugging Face (internet required once).



In [20]:
# Step 0d.5 — Embeddings for Ragas and Neo4jVector reconnect

from langchain_huggingface import HuggingFaceEmbeddings



EMBED_MODEL_NAME = os.getenv('RAG_EMBED_MODEL', 'sentence-transformers/all-MiniLM-L6-v2')

embeddings = HuggingFaceEmbeddings(

    model_name=EMBED_MODEL_NAME,

    encode_kwargs={'normalize_embeddings': True},

)

print('Embedding model:', EMBED_MODEL_NAME)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21459.04it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2


### Step 0e — Smoke test `ollama_model_runner.py`



Quick check that the runner, model name, and Ollama service work together **before** you run a full evaluation batch.



Terminal equivalent (from `Module_8/`):



```bash

python ollama_model_runner.py --host http://localhost:11434 \

  --models llama3.2:3b --prompt "Reply with exactly: Ollama OK" --max-tokens 32

```



If this fails, fix Ollama **before** Step 4 — a batch of 16+ LLM calls (8 questions × 2 pipelines) will fail the same way.



In [21]:
# Step 0e — Runner smoke test

if LLM_BACKEND == 'ollama':

    smoke = call_ollama_runner('Reply with exactly: Ollama OK', model=OLLAMA_MODEL)

    print('Smoke response:', smoke['response'][:120])

    print('Latency (sec):', round(smoke['latency_sec'], 3))

    print('Output tokens (eval_count):', smoke['eval_count'])

else:

    print('Skipped — OpenAI backend')



Smoke response: Ollama OK
Latency (sec): 0.285
Output tokens (eval_count): 5


**Expected output (Ollama):** A line containing `Ollama OK` (or similar), plus latency and `eval_count`.



If the reply is unexpected, verify `ollama list` shows your model and `OLLAMA_HOST` matches where `ollama serve` is listening.



---



# Step 1 — Verify lab graph and reconnect vector index



This evaluation assumes you already seeded the **`LangChainLab`** subgraph and built the vector index in **`Module_8_Using_Knowledge_Graph_with_LangChain.ipynb`**.



### What the companion notebook built



```text

(Port)-[:LOCATED_IN]->(Country)

(Organization)-[:OPERATES_IN]->(Port)

(Organization)-[:USES_ROUTE]->(Canal)

(Chunk)-[:MENTIONS]->(Entity)

```



Each `Chunk` node has a **`text`** property (embedded) and an **`embedding`** property (stored by `Neo4jVector.from_documents`).



### What Step 1 does



| Sub-step | Purpose |

|----------|---------|

| 1a | Connect `Neo4jGraph` for Cypher queries |

| 1b | Count lab nodes — fail fast if graph is empty |

| 1c | Reconnect existing vector index (no re-embedding) |

| 1d | Sanity-check retrieval on a known question |



### Step 1a — Connect `Neo4jGraph`



`Neo4jGraph` is the LangChain wrapper used throughout Module 8. It exposes:



- **`query(cypher, params)`** — run Cypher and get Python dict rows

- **`schema`** — text summary for text-to-Cypher (not needed in this evaluation notebook)



We reuse the same connection settings loaded from `.env` in Step 0.



In [22]:
# Step 1a — Connect Neo4jGraph

from langchain_neo4j import Neo4jGraph



neo4j_graph = Neo4jGraph(

    url=NEO4J_URI,

    username=NEO4J_USERNAME,

    password=NEO4J_PASSWORD,

    database=NEO4J_DATABASE,

)

print('Neo4jGraph connected.')



Neo4jGraph connected.


### Step 1b — Verify `LangChainLab` data exists



We count nodes by label. A healthy lab graph should include at minimum:



| Label | Typical count | Role |

|-------|---------------|------|

| `Chunk` | 4+ | Text passages for vector search |

| `Port` | 2 | Rotterdam, Singapore |

| `Organization` | 1 | Maersk |

| `Canal` | 1 | Panama Canal |

| `Country` | 2+ | Netherlands, Singapore |



If **`Chunk` count is 0**, stop here and run the companion notebook through Sections **3.1.3** and **3.2.1**.



**Verify in Neo4j Browser:**



```cypher

MATCH (o:Organization:LangChainLab)-[r]->(x)

RETURN o.id, type(r), x.id;

```



In [23]:
# Step 1b — Verify LangChainLab data exists

lab_counts = neo4j_graph.query('''

    MATCH (n:LangChainLab)

    RETURN labels(n)[0] AS label, count(*) AS cnt

    ORDER BY label

''')

print('LangChainLab node counts:')

for row in lab_counts:

    print(f"  {row['label']}: {row['cnt']}")



chunk_count = neo4j_graph.query(

    'MATCH (c:Chunk:LangChainLab) RETURN count(c) AS n'

)[0]['n']

if chunk_count < 1:

    raise RuntimeError(

        'No Chunk:LangChainLab nodes found. Run Module_8_Using_Knowledge_Graph_with_LangChain.ipynb '

        'through Section 3.1.3 and 3.2.1 first.'

    )

print('Chunk nodes ready:', chunk_count)



LangChainLab node counts:
  Canal: 1
  Country: 2
  Document: 1
  LangChainLab: 5
  Organization: 1
  Port: 2
Chunk nodes ready: 4


### Step 1c — Reconnect `Neo4jVector` index



In the LangChain lab you **created** the index with `Neo4jVector.from_documents`. Here we **reconnect** with `from_existing_index` so we:



- Do **not** re-embed all chunks (faster, idempotent)

- Use the same index name and node properties as the lab



| Parameter | Value in this lab |

|-----------|-------------------|

| `index_name` | `langchain_lab_chunk_index` |

| `node_label` | `Chunk` |

| `text_node_property` | `text` |

| `embedding_node_property` | `embedding` |



**If this fails:** the vector index may not exist yet — run Section **3.2.1c** of the companion notebook first.



We wrap the store with **`as_retriever(search_kwargs={'k': RETRIEVAL_K})`** so both pipelines retrieve the same number of chunks.



In [24]:
# Step 1c — Reconnect Neo4jVector index (read existing embeddings)

from langchain_neo4j import Neo4jVector



VECTOR_INDEX_NAME = 'langchain_lab_chunk_index'

VECTOR_NODE_LABEL = 'Chunk'

VECTOR_TEXT_PROPERTY = 'text'

VECTOR_EMBEDDING_PROPERTY = 'embedding'



vector_store = Neo4jVector.from_existing_index(

    embeddings,

    url=NEO4J_URI,

    username=NEO4J_USERNAME,

    password=NEO4J_PASSWORD,

    database=NEO4J_DATABASE,

    index_name=VECTOR_INDEX_NAME,

    node_label=VECTOR_NODE_LABEL,

    text_node_property=VECTOR_TEXT_PROPERTY,

    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,

)

vector_retriever = vector_store.as_retriever(search_kwargs={'k': RETRIEVAL_K})

print('Vector index connected:', VECTOR_INDEX_NAME, '| k =', RETRIEVAL_K)



Vector index connected: langchain_lab_chunk_index | k = 3


### Step 1d — Quick retrieval sanity check



We ask a question whose answer is clearly in **`chunk_rotterdam`**: *"Which port is the largest in Europe?"*



**What to look for:**



- Top hit should mention **Rotterdam** and **Europe**.

- Metadata should include `chunk_id` (required for GraphRAG expansion in Step 2).



If retrieval returns unrelated chunks, re-run vector index creation in the companion notebook or check that chunk `text` properties are populated.



In [25]:
# Step 1d — Quick retrieval sanity check

query = 'Which port is the largest in Europe?'

hits = vector_retriever.invoke(query)

print('Query:', query)

for i, doc in enumerate(hits, 1):

    print(f"{i}. {doc.metadata.get('chunk_id', '?')} -> {doc.page_content[:80]}...")



Query: Which port is the largest in Europe?
1. chunk_rotterdam -> The Port of Rotterdam is the largest port in Europe and a major hub for containe...
2. chunk_singapore -> The Port of Singapore is a leading transshipment hub in Southeast Asia....
3. chunk_maersk -> Maersk is a Danish shipping company that operates vessels and uses routes such a...


**Checkpoint:** Retrieval works and `chunk_id` appears in metadata. You are ready to build evaluation pipelines in Step 2.



---



# Step 2 — Build vector RAG and GraphRAG answer functions



Evaluation requires **two comparable pipelines** that return the same dictionary shape so we can batch-run them and feed RAGAS.



### Output dictionary (both pipelines)



| Field | Purpose |

|-------|---------|

| `answer` | Model response text |

| `contexts` | List of strings — RAGAS **faithfulness** checks claims against these |

| `retrieval_sec` | Time for vector search (+ graph Cypher for GraphRAG) |

| `generation_sec` | Time for LLM answer generation |

| `total_sec` | Sum — used for latency comparison in Step 6 |

| `eval_count` | Ollama output tokens — **cost proxy** |



### Why compare two pipelines?



```text

Vector RAG:   question → embed → top-k chunks → LLM → answer

GraphRAG:     question → embed → top-k chunks → Cypher expansion → LLM → answer

                                              ↑

                                    MENTIONS, USES_ROUTE, ...

```



- **Vector RAG** is the baseline — answers from chunk text only.

- **GraphRAG** adds explicit relationship facts that may not appear verbatim in a chunk (e.g. `Maersk -[:USES_ROUTE]-> Panama_Canal`).



GraphRAG may improve **relationship-heavy** questions at the cost of extra Cypher latency and longer prompts (more tokens).



### Step 2a — Shared prompt templates



We use **strict grounding prompts** so models say *"I do not have enough information"* when context is insufficient — this makes **faithfulness** scores easier to interpret.



| Template | Used by | Context blocks |

|----------|---------|----------------|

| `RAG_PROMPT_TEMPLATE` | Vector RAG | Joined chunk text |

| `GRAPHRAG_PROMPT_TEMPLATE` | GraphRAG | Chunk text **and** graph fact lines |



`format_docs()` converts a list of LangChain `Document` objects into bullet-point text for the prompt.



In [26]:
# Step 2a — Shared prompt template and format helper

RAG_PROMPT_TEMPLATE = '''

You are a strict assistant for a maritime knowledge lab.

Answer using ONLY the provided context.

If the context does not contain enough information, say: "I do not have enough information from the provided context."



Question:

{question}



Context:

{context}

'''.strip()



GRAPHRAG_PROMPT_TEMPLATE = '''

You are a strict assistant for a maritime knowledge lab.

Answer using ONLY the chunk text and graph facts below.

If the context does not contain enough information, say: "I do not have enough information from the provided context."



Question:

{question}



Chunk text:

{chunks}



Graph facts:

{graph}

'''.strip()





def format_docs(docs):

    return '\n\n'.join(f'- {d.page_content}' for d in docs)



### Step 2b — Graph expansion helpers



These functions mirror **Section 3.2.3** of the LangChain lab notebook.



**`graph_context_for_chunks(chunk_ids)`** runs Cypher:



1. Start from retrieved chunk IDs.

2. Follow `(Chunk)-[:MENTIONS]->(Entity)`.

3. Optionally expand one hop on `LOCATED_IN`, `OPERATES_IN`, `USES_ROUTE`.

4. Format each row as a readable line for the LLM.



**`graphrag_retrieve(question)`** orchestrates:



1. Vector search (same retriever as vector RAG).

2. Extract `chunk_id` from each document's metadata.

3. Call graph expansion and return `{chunks, graph_context}`.



**Production note:** This is a teaching helper. Production GraphRAG may use packaged retrievers (e.g. `neo4j-graphrag`) with ranking, limits, and security filters.



In [27]:
# Step 2b — Graph expansion helpers (from LangChain lab Section 3.2.3)

def graph_context_for_chunks(chunk_ids: list[str]) -> str:

    rows = neo4j_graph.query(

        '''

        UNWIND $ids AS chunk_id

        MATCH (c:Chunk:LangChainLab {id: chunk_id})-[:MENTIONS]->(e:LangChainLab)

        OPTIONAL MATCH (e)-[r]-(n:LangChainLab)

        WHERE type(r) IN ['LOCATED_IN', 'OPERATES_IN', 'USES_ROUTE']

        RETURN DISTINCT c.id AS chunk, e.id AS entity, type(r) AS rel, n.id AS related

        LIMIT 30

        ''',

        params={'ids': chunk_ids},

    )

    lines = []

    for row in rows:

        lines.append(

            f"Chunk {row.get('chunk')} mentions {row.get('entity')}; "

            f"{row.get('rel')} -> {row.get('related')}"

        )

    return '\n'.join(lines) if lines else '(no graph context)'





def graphrag_retrieve(question: str, k: int | None = None) -> dict:

    k = k or RETRIEVAL_K

    docs = vector_retriever.invoke(question)[:k]

    chunk_ids = [d.metadata.get('chunk_id') for d in docs if d.metadata.get('chunk_id')]

    return {

        'chunks': docs,

        'graph_context': graph_context_for_chunks(chunk_ids),

    }



### Step 2c — Vector-only RAG answer function



**Flow for each question:**



1. **Retrieve** — `vector_retriever.invoke(question)` → list of `Document`

2. **Build context** — join `page_content` strings with `---` separators

3. **Generate** — format prompt, call Ollama runner (or OpenAI)

4. **Return** — answer, contexts, timing, token count



We time **retrieval** and **generation** separately so Step 6 can show whether GraphRAG's extra Cypher adds retrieval overhead vs. longer generation from bigger prompts.



In [28]:
# Step 2c — Vector-only RAG answer function

def answer_with_vector_rag(question: str) -> dict:

    t_ret0 = time.perf_counter()

    retrieved_docs = vector_retriever.invoke(question)

    retrieval_sec = time.perf_counter() - t_ret0



    contexts = [d.page_content for d in retrieved_docs]

    joined_context = '\n\n---\n\n'.join(contexts)

    prompt = RAG_PROMPT_TEMPLATE.format(question=question, context=joined_context)



    if LLM_BACKEND == 'ollama':

        gen = call_ollama_runner(prompt)

        answer_text = gen['response']

        generation_sec = gen['latency_sec']

        eval_count = gen['eval_count']

    else:

        t_gen0 = time.perf_counter()

        answer_text = chat_llm.invoke(prompt).content.strip()

        generation_sec = time.perf_counter() - t_gen0

        eval_count = 0



    return {

        'question': question,

        'answer': answer_text,

        'contexts': contexts,

        'retrieval_sec': retrieval_sec,

        'generation_sec': generation_sec,

        'total_sec': retrieval_sec + generation_sec,

        'eval_count': eval_count,

        'pipeline': 'vector_rag',

    }



### Step 2d — GraphRAG answer function



Same structure as vector RAG, with two differences:



1. **Retrieval** calls `graphrag_retrieve()` (vector + Cypher).

2. **Contexts for RAGAS** include chunk text **plus** a `[Graph facts]` block so faithfulness can verify relationship claims against retrieved evidence.



The LLM prompt has separate **`chunks`** and **`graph`** sections — mirroring how humans read unstructured text and structured facts side by side.



**Important for RAGAS:** If graph facts are omitted from `contexts`, faithfulness may penalize correct relationship answers that only appear in the graph block.



In [29]:
# Step 2d — GraphRAG answer function

def answer_with_graphrag(question: str) -> dict:

    t_ret0 = time.perf_counter()

    pack = graphrag_retrieve(question)

    retrieval_sec = time.perf_counter() - t_ret0



    chunks_txt = format_docs(pack['chunks'])

    chunk_contexts = [d.page_content for d in pack['chunks']]

    graph_ctx = pack['graph_context']

    # Ragas faithfulness checks against retrieved contexts; include graph facts as an extra context block.

    contexts = chunk_contexts + ([f'[Graph facts]\n{graph_ctx}'] if graph_ctx else [])



    prompt = GRAPHRAG_PROMPT_TEMPLATE.format(

        question=question,

        chunks=chunks_txt,

        graph=graph_ctx,

    )



    if LLM_BACKEND == 'ollama':

        gen = call_ollama_runner(prompt)

        answer_text = gen['response']

        generation_sec = gen['latency_sec']

        eval_count = gen['eval_count']

    else:

        t_gen0 = time.perf_counter()

        answer_text = chat_llm.invoke(prompt).content.strip()

        generation_sec = time.perf_counter() - t_gen0

        eval_count = 0



    return {

        'question': question,

        'answer': answer_text,

        'contexts': contexts,

        'retrieval_sec': retrieval_sec,

        'generation_sec': generation_sec,

        'total_sec': retrieval_sec + generation_sec,

        'eval_count': eval_count,

        'pipeline': 'graphrag',

    }



### Step 2e — Smoke test both pipelines



We use a **relationship-heavy** question: *"How is Maersk connected to the Panama Canal?"*



This question is ideal for a quick A/B check:



| Pipeline | What you might see |

|----------|-------------------|

| Vector RAG | May answer from Maersk or Panama chunk text alone |

| GraphRAG | Should cite `USES_ROUTE` or equivalent graph fact |



Compare **answer quality**, **total_sec**, and **eval_count** before running the full 8-question batch.



In [30]:
# Step 2e — Smoke test both pipelines on one question

_demo_q = 'How is Maersk connected to the Panama Canal?'

_demo_vec = answer_with_vector_rag(_demo_q)

_demo_graph = answer_with_graphrag(_demo_q)

print('Question:', _demo_q)

print('\n--- Vector RAG ---')

print(_demo_vec['answer'][:400])

print('Total sec:', round(_demo_vec['total_sec'], 3), '| eval_count:', _demo_vec['eval_count'])

print('\n--- GraphRAG ---')

print(_demo_graph['answer'][:400])

print('Total sec:', round(_demo_graph['total_sec'], 3), '| eval_count:', _demo_graph['eval_count'])



Question: How is Maersk connected to the Panama Canal?

--- Vector RAG ---
Since Maersk operates vessels and uses routes such as the Panama Canal, it can be inferred that Maersk is connected to the Panama Canal through its use of this route for transportation purposes.
Total sec: 0.514 | eval_count: 40

--- GraphRAG ---
Maersk is connected to the Panama Canal through a route it uses.
Total sec: 0.38 | eval_count: 16


**How to read the smoke test:**



- If GraphRAG is **slower** but **more accurate** on this question, that supports the graph-augmentation trade-off.

- If both answers are similar and GraphRAG is slower, graph expansion may not help for your question set — Step 6 quantifies this across all rows.

- If either pipeline errors, fix before Step 4 (batch runs do not stop on first failure by default).



---



# Step 3 — Ground-truth dataset



A **ground truth** row pairs a **`question`** with a **`ground_truth`** reference answer. RAGAS compares the model's **`answer`** to **`ground_truth`** (answer relevancy) and to **`contexts`** (faithfulness).



### Why manual labels for this lab?



The maritime lab graph is **small and auditable** — you can verify every reference answer in Neo4j Browser or the companion notebook seed cells. Manual labels avoid **circular bias** (the same model grading itself).



For larger corpora, see **`Module_5_Building_an_End-to-End_RAG_Evaluation.ipynb`** (Module 4) for LLM-synthetic ground-truth workflows.



### Column reference



| Column | Meaning |

|--------|---------|

| `question` | What you ask both pipelines |

| `ground_truth` | Short factual reference answer |

| `needs_graph` | `True` if correct answers likely need relationship traversal |



Questions with **`needs_graph=True`** are the best signal for whether GraphRAG earns its extra latency.



### Step 3a — Manual ground-truth rows



The 8 questions below cover:



- **Chunk-only facts** — answerable from vector retrieval alone (`needs_graph=False`)

- **Graph facts** — ports in countries, Maersk operations, canal routes (`needs_graph=True`)



When extending the lab (e.g. after Section 3.2.4 corpus import), add rows that reflect **new** chunks and entities — keep `ground_truth` strings short and verifiable.



In [31]:
# Step 3a — Manual ground-truth rows for the LangChain lab graph

manual_ground_truth_data = [

    {

        'question': 'Which port is the largest in Europe according to the lab data?',

        'ground_truth': 'The Port of Rotterdam is the largest port in Europe.',

        'needs_graph': False,

    },

    {

        'question': 'Which country is the Port of Singapore located in?',

        'ground_truth': 'The Port of Singapore is located in Singapore.',

        'needs_graph': True,

    },

    {

        'question': 'Which organization operates in the Port of Rotterdam?',

        'ground_truth': 'Maersk operates in the Port of Rotterdam.',

        'needs_graph': True,

    },

    {

        'question': 'What route does Maersk use according to the graph?',

        'ground_truth': 'Maersk uses the Panama Canal route.',

        'needs_graph': True,

    },

    {

        'question': 'How is Maersk connected to the Panama Canal?',

        'ground_truth': 'Maersk is connected to the Panama Canal through a USES_ROUTE relationship.',

        'needs_graph': True,

    },

    {

        'question': 'What do the Panama Canal connect?',

        'ground_truth': 'The Panama Canal connects the Atlantic and Pacific oceans.',

        'needs_graph': False,

    },

    {

        'question': 'Which port is described as a leading transshipment hub in Southeast Asia?',

        'ground_truth': 'The Port of Singapore is a leading transshipment hub in Southeast Asia.',

        'needs_graph': False,

    },

    {

        'question': 'What country is Maersk associated with in the lab graph?',

        'ground_truth': 'Maersk is a Danish shipping company (Denmark).',

        'needs_graph': False,

    },

]



ground_truth_df = pd.DataFrame(manual_ground_truth_data)

print('Ground-truth rows:', len(ground_truth_df))

ground_truth_df



Ground-truth rows: 8


,question,ground_truth,needs_graph
0,Which port is the largest in Europe according to the lab data?,The Port of Rotterdam is the largest port in Europe.,False
1,Which country is the Port of Singapore located in?,The Port of Singapore is located in Singapore.,True
2,Which organization operates in the Port of Rotterdam?,Maersk operates in the Port of Rotterdam.,True
3,What route does Maersk use according to the graph?,Maersk uses the Panama Canal route.,True
4,How is Maersk connected to the Panama Canal?,Maersk is connected to the Panama Canal through a USES_ROUTE relationship.,True
5,What do the Panama Canal connect?,The Panama Canal connects the Atlantic and Pacific oceans.,False
6,Which port is described as a leading transshipment hub in Southeast Asia?,The Port of Singapore is a leading transshipment hub in Southeast Asia.,False
7,What country is Maersk associated with in the lab graph?,Maersk is a Danish shipping company (Denmark).,False


### Step 3b — Save ground truth to CSV



Persisting labels to CSV lets you:



- Re-run Steps 4–6 without re-editing Python lists

- Share the evaluation set with teammates

- Version-control question sets (without answers in git if preferred)



The file is written to **`outputs/eval_graphrag/ground_truth_graphrag_lab.csv`**.



In [32]:
# Step 3b — Save ground truth to CSV (reusable across reruns)

gt_path = OUTPUT_DIR / 'ground_truth_graphrag_lab.csv'

ground_truth_df.to_csv(gt_path, index=False)

print('Saved:', gt_path)



Saved: /home/ethan/newgen/KMOU_Course/Module_8/outputs/eval_graphrag/ground_truth_graphrag_lab.csv


---



# Step 4 — Run evaluation for both pipelines



Now we run **every ground-truth question** through both pipelines and collect raw outputs for RAGAS.



### Batch design



| Step | Pipeline | Rows |

|------|----------|------|

| 4b | Vector RAG | one row per question |

| 4c | GraphRAG | one row per question |



Each row stores **answer**, **contexts**, **timing**, and **eval_count** so Steps 5–6 never re-call the LLM for the same question (unless you re-run this step).



### Time budget



With 8 questions × 2 pipelines = **16 answer generations**, plus RAGAS LLM calls in Step 5:



- Local Ollama (`llama3.2:3b`): roughly **5–15 minutes** total depending on hardware

- Reduce the ground-truth list while debugging



> **Tip:** Run Step 4b alone first. If vector RAG works, proceed to 4c.



### Step 4a — Batch runner helper



`run_pipeline_eval()` loops over ground-truth rows with **`tqdm`** progress bars.



It accepts any **`answer_fn`** with the same signature as `answer_with_vector_rag` / `answer_with_graphrag`, keeping the batch logic **pipeline-agnostic**.



The output DataFrame is ready for RAGAS (Step 5) after minor column selection.



In [33]:
# Step 4a — Batch runner helper

def run_pipeline_eval(gt_rows: list[dict], answer_fn, pipeline_name: str) -> pd.DataFrame:

    records = []

    for row in tqdm(gt_rows, desc=f'Eval {pipeline_name}'):

        out = answer_fn(row['question'])

        records.append({

            'question': row['question'],

            'ground_truth': row['ground_truth'],

            'needs_graph': row.get('needs_graph', False),

            'answer': out['answer'],

            'contexts': out['contexts'],

            'retrieval_sec': out['retrieval_sec'],

            'generation_sec': out['generation_sec'],

            'total_sec': out['total_sec'],

            'eval_count': out['eval_count'],

            'pipeline': pipeline_name,

        })

    return pd.DataFrame(records)





gt_rows = ground_truth_df.to_dict('records')



### Step 4b — Run vector RAG evaluation



This is the **baseline** run. Inspect the preview table:



- **`answer`** — does it match `ground_truth` roughly?

- **`total_sec`** — typical latency per question on your machine

- **`eval_count`** — output tokens (baseline cost proxy)



Unusually slow rows may indicate long retrieved context or model cold-start on first call.



In [34]:
# Step 4b — Run vector RAG evaluation

eval_df_vector = run_pipeline_eval(gt_rows, answer_with_vector_rag, 'vector_rag')

print('Vector RAG rows:', len(eval_df_vector))

eval_df_vector[['question', 'answer', 'total_sec', 'eval_count']].head()



Eval vector_rag: 100%|██████████| 8/8 [00:03<00:00,  2.28it/s]

Vector RAG rows: 8


,question,answer,total_sec,eval_count
0,Which port is the largest in Europe according to the lab data?,I do not have enough information from the provided context to determine which port is the largest in Europe accordin...,0.562513,42
1,Which country is the Port of Singapore located in?,I do not have enough information from the provided context to determine which country the Port of Singapore is locat...,0.421274,23
2,Which organization operates in the Port of Rotterdam?,I do not have enough information from the provided context to determine which organization operates in the Port of R...,0.411786,22
3,What route does Maersk use according to the graph?,I do not have enough information from the provided context to determine which specific route Maersk uses according t...,0.576044,43
4,How is Maersk connected to the Panama Canal?,"Since Maersk operates vessels and uses routes such as the Panama Canal, it can be inferred that Maersk is connected ...",0.547956,40


### Step 4c — Run GraphRAG evaluation



Same questions, same `RETRIEVAL_K`, but with graph expansion.



Compare preview columns to Step 4b:



- Is **`total_sec`** consistently higher? (expected — extra Cypher + longer prompts)

- Is **`eval_count`** higher? (longer graph context → more input tokens indirectly; output may also grow)



Do **not** interpret accuracy yet — RAGAS scores that in Step 5.



In [35]:
# Step 4c — Run GraphRAG evaluation

eval_df_graphrag = run_pipeline_eval(gt_rows, answer_with_graphrag, 'graphrag')

print('GraphRAG rows:', len(eval_df_graphrag))

eval_df_graphrag[['question', 'answer', 'total_sec', 'eval_count']].head()



Eval graphrag: 100%|██████████| 8/8 [00:03<00:00,  2.43it/s]

GraphRAG rows: 8


,question,answer,total_sec,eval_count
0,Which port is the largest in Europe according to the lab data?,I do not have enough information from the provided context to determine which port is the largest in Europe accordin...,0.659003,53
1,Which country is the Port of Singapore located in?,The country where the Port of Singapore is located is Singapore.,0.377807,13
2,Which organization operates in the Port of Rotterdam?,I do not have enough information from the provided context to determine which organization operates in the Port of R...,0.404780,22
3,What route does Maersk use according to the graph?,"According to the graph facts, Maersk uses the Panama Canal route.",0.386686,16
4,How is Maersk connected to the Panama Canal?,Maersk is connected to the Panama Canal through a route it uses.,0.370693,16


### Step 4d — Combine and save raw outputs



We merge both pipelines into one CSV for audit trails. Context lists are flattened with `---` separators for spreadsheet readability.



**File:** `outputs/eval_graphrag/graphrag_eval_raw_outputs.csv`



Keep this file when comparing pipeline versions over time (e.g. before/after increasing `EVAL_RETRIEVAL_K`).



In [36]:
# Step 4d — Combine and save raw outputs

eval_df_all = pd.concat([eval_df_vector, eval_df_graphrag], ignore_index=True)

raw_path = OUTPUT_DIR / 'graphrag_eval_raw_outputs.csv'

export_df = eval_df_all.copy()

export_df['contexts'] = export_df['contexts'].apply(lambda xs: '\n\n---\n\n'.join(xs))

export_df.to_csv(raw_path, index=False)

print('Saved:', raw_path)



Saved: /home/ethan/newgen/KMOU_Course/Module_8/outputs/eval_graphrag/graphrag_eval_raw_outputs.csv


---



# Step 5 — Score with RAGAS



[RAGAS](https://docs.ragas.io/) (Retrieval Augmented Generation Assessment) provides reference-free and reference-based metrics for RAG systems.



### Metrics used in this lab



| Metric | Range | Question it answers |

|--------|-------|---------------------|

| **Faithfulness** | 0–1 | Is the answer **supported by retrieved context**? |

| **Answer relevancy** | 0–1 | Does the answer **address the question** (vs. ground truth intent)? |



### How to interpret score combinations



| Faithfulness | Relevancy | Typical meaning |

|--------------|-----------|-----------------|

| High | High | Healthy — grounded and on-topic |

| High | Low | Grounded but misses question intent |

| Low | High | Sounds right but **not supported** by context (hallucination risk) |

| Low | Low | Broad pipeline issues (retrieval, prompt, or model) |



### RAGAS input schema



Required columns: `question`, `answer`, `contexts` (list of strings), `ground_truth`.



> **Note:** RAGAS makes **additional LLM calls** per row (and uses embeddings for answer relevancy). Budget **extra time** beyond Step 4.



### Step 5a.0 — RAGAS compatibility shim (run before importing RAGAS)



**Known issue:** `ragas` 0.4.x imports `ChatVertexAI` from a **removed** path in modern `langchain-community`:



```text

langchain_community.chat_models.vertexai  # removed in langchain-community 0.4+

```



This lab uses **Ollama** only — we do not need Google Vertex AI. The next cell registers a small **stub module** so `import ragas` succeeds.



| Symptom | Fix |

|---------|-----|

| `No module named 'langchain_community.chat_models.vertexai'` | Run the shim cell below, then Step 5a |

| Shim cell already ran in this kernel | Safe to re-run |



Upstream reference: [ragas issue #2745](https://github.com/vibrantlabsai/ragas/issues/2745).



In [37]:
# Step 5a.0 — Compatibility shim for ragas + langchain-community 0.4+

import sys

import types



import langchain_community.llms as _lc_llms





class _VertexStub:

    """Optional VertexAI symbols for ragas import only (not used in this lab)."""





_vertexai_mod = types.ModuleType('langchain_community.chat_models.vertexai')

_vertexai_mod.ChatVertexAI = _VertexStub

sys.modules['langchain_community.chat_models.vertexai'] = _vertexai_mod



if not hasattr(_lc_llms, 'VertexAI'):

    _lc_llms.VertexAI = _VertexStub



print('RAGAS compatibility shim applied.')



RAGAS compatibility shim applied.


/tmp/ipykernel_1200312/1244935741.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.llms as _lc_llms


### Step 5a — Import RAGAS



Run **Step 5a.0** first if you have not already applied the compatibility shim in this kernel session.



We import:



- **`evaluate`** — main RAGAS scoring function

- **`faithfulness`** and **`answer_relevancy`** — metrics used in this lab



Deprecation warnings about `ragas.metrics.collections` are expected on ragas 0.4.x and can be ignored for this course.



In [38]:
# Step 5a — Import RAGAS

from datasets import Dataset

from ragas import evaluate

from ragas.metrics import faithfulness, answer_relevancy



print('RAGAS imported successfully.')



RAGAS imported successfully.


/tmp/ipykernel_1200312/383131074.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_1200312/383131074.py:7: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy


### Step 5b — RAGAS helper function



`run_ragas_scores()`:



1. Selects the four columns RAGAS expects.

2. Converts to Hugging Face **`Dataset`**.

3. Calls **`evaluate()`** with `faithfulness` and `answer_relevancy`.

4. Joins scores back onto the original evaluation DataFrame.



We pass the same **`ragas_llm`** and **`embeddings`** configured in Step 0 so scoring is reproducible.



In [39]:
# Step 5b — Ragas helper

def run_ragas_scores(eval_df: pd.DataFrame, label: str):

    ragas_ready = eval_df[['question', 'answer', 'contexts', 'ground_truth']].copy()

    ragas_dataset = Dataset.from_pandas(ragas_ready, preserve_index=False)

    print(f'RAGAS dataset ({label}):', ragas_dataset)

    result = evaluate(

        dataset=ragas_dataset,

        metrics=[faithfulness, answer_relevancy],

        llm=ragas_llm,

        embeddings=embeddings,

    )

    score_df = result.to_pandas()

    merged = eval_df.reset_index(drop=True).join(score_df[['faithfulness', 'answer_relevancy']])

    return merged, score_df



### Step 5c — Score vector RAG



This establishes the **baseline** accuracy numbers.



Focus on:



- **Mean faithfulness** — how often vector RAG stays within retrieved chunks

- **Mean answer relevancy** — how well answers align with `ground_truth` intent

- Rows where **`needs_graph=True`** but scores are low — candidates for GraphRAG improvement in Step 5d



In [40]:
# Step 5c — Score vector RAG

scored_vector, _ = run_ragas_scores(eval_df_vector, 'vector_rag')

print('Vector RAG mean faithfulness:', round(scored_vector['faithfulness'].mean(), 3))

print('Vector RAG mean answer_relevancy:', round(scored_vector['answer_relevancy'].mean(), 3))

scored_vector[['question', 'faithfulness', 'answer_relevancy', 'needs_graph']].head()



RAGAS dataset (vector_rag): Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 8
})


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   6%|▋         | 1/16 [00:37<09:15, 37.02s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  94%|█████████▍| 15/16 [02:06<00:05,  5.55s/it]Exception raised in Job[14]: OutputParserException(Invalid json output: The output string did not satisfy the constraints given in the prompt. Fix the output string and return i

Vector RAG mean faithfulness: 0.548
Vector RAG mean answer_relevancy: 0.3


,question,faithfulness,answer_relevancy,needs_graph
0,Which port is the largest in Europe according to the lab data?,0.500000,0.0,False
1,Which country is the Port of Singapore located in?,0.333333,0.0,True
2,Which organization operates in the Port of Rotterdam?,0.000000,0.0,True
3,What route does Maersk use according to the graph?,0.750000,0.0,True
4,How is Maersk connected to the Panama Canal?,0.500000,1.0,True


### Step 5d — Score GraphRAG



Compare means to Step 5c:



- **Higher faithfulness on `needs_graph=True` rows** → graph facts helped grounding

- **Higher answer relevancy on relationship questions** → graph context improved factual completeness

- **Lower faithfulness overall** → model may be inventing claims from graph lines not in `contexts` (check Step 2d context assembly)



Slice the preview by **`needs_graph`** mentally before reading Step 6 aggregates.



In [41]:
# Step 5d — Score GraphRAG

scored_graphrag, _ = run_ragas_scores(eval_df_graphrag, 'graphrag')

print('GraphRAG mean faithfulness:', round(scored_graphrag['faithfulness'].mean(), 3))

print('GraphRAG mean answer_relevancy:', round(scored_graphrag['answer_relevancy'].mean(), 3))

scored_graphrag[['question', 'faithfulness', 'answer_relevancy', 'needs_graph']].head()



RAGAS dataset (graphrag): Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 8
})


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   6%|▋         | 1/16 [00:52<13:09, 52.65s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 16/16 [02:15<00:00,  8.49s/it]


GraphRAG mean faithfulness: 0.667
GraphRAG mean answer_relevancy: 0.567


,question,faithfulness,answer_relevancy,needs_graph
0,Which port is the largest in Europe according to the lab data?,0.333333,0.000000,False
1,Which country is the Port of Singapore located in?,1.000000,0.972028,True
2,Which organization operates in the Port of Rotterdam?,0.500000,0.000000,True
3,What route does Maersk use according to the graph?,1.000000,0.891971,True
4,How is Maersk connected to the Panama Canal?,0.500000,1.000000,True


### Step 5e — Save scored tables



| File | Contents |

|------|----------|

| `scored_vector_rag.csv` | Vector RAG answers + RAGAS scores |

| `scored_graphrag.csv` | GraphRAG answers + RAGAS scores |



These are the primary inputs for Step 6 comparison and Step 7 failure analysis.



In [42]:
# Step 5e — Save scored tables

scored_vector.to_csv(OUTPUT_DIR / 'scored_vector_rag.csv', index=False)

scored_graphrag.to_csv(OUTPUT_DIR / 'scored_graphrag.csv', index=False)

print('Saved scored CSVs under', OUTPUT_DIR)



Saved scored CSVs under /home/ethan/newgen/KMOU_Course/Module_8/outputs/eval_graphrag


---



# Step 6 — Compare time, cost, and accuracy



Production teams rarely optimize **accuracy alone**. This step combines three dimensions:



### Comparison dimensions



| Dimension | Vector RAG | GraphRAG | Why it matters |

|-----------|------------|----------|----------------|

| **Accuracy** | Mean faithfulness & relevancy | Same | Quality |

| **Latency** | Mean / p95 `total_sec` | Same | User experience |

| **Cost proxy** | Mean / total `eval_count` | Same | Capacity planning |



For **hosted APIs**, multiply token counts by your provider's price sheet. For **local Ollama**, dollar cost is ~$0, but **tokens and seconds** still predict hardware load and batch job duration.



We also compute metrics on **`needs_graph=True`** subsets — the fairest place to expect GraphRAG wins.



### Step 6a — Accuracy summary



`metric_summary()` aggregates:



- Overall mean **faithfulness** and **answer_relevancy**

- Same means restricted to **`needs_graph=True`** rows



If GraphRAG only wins on the subset but ties overall, graph expansion helps **relationship questions** specifically — a common real-world pattern.



In [43]:
# Step 6a — Accuracy summary

def metric_summary(scored_df: pd.DataFrame, pipeline: str) -> dict:

    return {

        'pipeline': pipeline,

        'faithfulness_mean': float(scored_df['faithfulness'].mean()),

        'answer_relevancy_mean': float(scored_df['answer_relevancy'].mean()),

        'faithfulness_mean_needs_graph': float(scored_df.loc[scored_df['needs_graph'], 'faithfulness'].mean())

        if scored_df['needs_graph'].any() else None,

        'answer_relevancy_mean_needs_graph': float(scored_df.loc[scored_df['needs_graph'], 'answer_relevancy'].mean())

        if scored_df['needs_graph'].any() else None,

    }



accuracy_summary = pd.DataFrame([

    metric_summary(scored_vector, 'vector_rag'),

    metric_summary(scored_graphrag, 'graphrag'),

])

accuracy_summary



,pipeline,faithfulness_mean,answer_relevancy_mean,faithfulness_mean_needs_graph,answer_relevancy_mean_needs_graph
0,vector_rag,0.547619,0.300153,0.395833,0.250
1,graphrag,0.666667,0.566956,0.750000,0.716


### Step 6b — Latency and cost summary



| Metric | Meaning |

|--------|---------|

| `mean_total_sec` | Average end-to-end time per question |

| `p95_total_sec` | 95th percentile — captures slow outliers |

| `mean_retrieval_sec` | Vector (+ graph) retrieval only |

| `mean_generation_sec` | LLM call only |

| `mean_eval_count` | Average output tokens per answer |

| `total_eval_count` | Sum across all questions — batch cost proxy |



GraphRAG often shows **higher `mean_retrieval_sec`** (Cypher) and **higher `mean_eval_count`** (longer prompts).



In [44]:
# Step 6b — Latency and cost summary

def ops_summary(eval_df: pd.DataFrame, pipeline: str) -> dict:

    return {

        'pipeline': pipeline,

        'mean_total_sec': float(eval_df['total_sec'].mean()),

        'p95_total_sec': float(eval_df['total_sec'].quantile(0.95)),

        'mean_retrieval_sec': float(eval_df['retrieval_sec'].mean()),

        'mean_generation_sec': float(eval_df['generation_sec'].mean()),

        'mean_eval_count': float(eval_df['eval_count'].mean()),

        'total_eval_count': int(eval_df['eval_count'].sum()),

    }



ops_df = pd.DataFrame([

    ops_summary(eval_df_vector, 'vector_rag'),

    ops_summary(eval_df_graphrag, 'graphrag'),

])

ops_df



,pipeline,mean_total_sec,p95_total_sec,mean_retrieval_sec,mean_generation_sec,mean_eval_count,total_eval_count
0,vector_rag,0.437033,0.571308,0.009080,0.427953,24.75,198
1,graphrag,0.410796,0.570025,0.012311,0.398485,20.00,160


### Step 6c — Combined comparison table



Derived columns:



| Column | Meaning |

|--------|---------|

| `faithfulness_delta_vs_vector` | GraphRAG mean faithfulness minus vector RAG (positive = graph wins) |

| `latency_ratio_vs_vector` | GraphRAG mean latency divided by vector RAG (>1 = graph is slower) |



**Decision framing:** Accept GraphRAG if accuracy gains on important question types exceed your latency/cost budget (e.g. +0.1 faithfulness worth 1.3× latency).



In [45]:
# Step 6c — Combined comparison table

comparison = accuracy_summary.merge(ops_df, on='pipeline')

comparison['faithfulness_delta_vs_vector'] = comparison['faithfulness_mean'] - comparison.loc[

    comparison['pipeline'] == 'vector_rag', 'faithfulness_mean'

].iloc[0]

comparison['latency_ratio_vs_vector'] = comparison['mean_total_sec'] / comparison.loc[

    comparison['pipeline'] == 'vector_rag', 'mean_total_sec'

].iloc[0]

comparison



,pipeline,faithfulness_mean,answer_relevancy_mean,faithfulness_mean_needs_graph,answer_relevancy_mean_needs_graph,mean_total_sec,p95_total_sec,mean_retrieval_sec,mean_generation_sec,mean_eval_count,total_eval_count,faithfulness_delta_vs_vector,latency_ratio_vs_vector
0,vector_rag,0.547619,0.300153,0.395833,0.250,0.437033,0.571308,0.009080,0.427953,24.75,198,0.000000,1.000000
1,graphrag,0.666667,0.566956,0.750000,0.716,0.410796,0.570025,0.012311,0.398485,20.00,160,0.119048,0.939966


### Step 6d — Side-by-side per question



This table is the most actionable view for learners:



- **`faithfulness_gain`** — GraphRAG minus vector RAG per question

- **`latency_delta_sec`** — extra seconds GraphRAG cost on that question



Sort by **`faithfulness_gain`** descending to see where graph augmentation helped most. Cross-check with **`needs_graph`** — gains on `True` rows validate the GraphRAG design.



In [46]:
# Step 6d — Side-by-side per question (accuracy + latency)

side_by_side = scored_vector[['question', 'needs_graph', 'faithfulness', 'answer_relevancy', 'total_sec']].rename(

    columns={

        'faithfulness': 'vec_faithfulness',

        'answer_relevancy': 'vec_answer_relevancy',

        'total_sec': 'vec_total_sec',

    }

).merge(

    scored_graphrag[['question', 'faithfulness', 'answer_relevancy', 'total_sec']].rename(

        columns={

            'faithfulness': 'graph_faithfulness',

            'answer_relevancy': 'graph_answer_relevancy',

            'total_sec': 'graph_total_sec',

        }

    ),

    on='question',

)

side_by_side['faithfulness_gain'] = side_by_side['graph_faithfulness'] - side_by_side['vec_faithfulness']

side_by_side['latency_delta_sec'] = side_by_side['graph_total_sec'] - side_by_side['vec_total_sec']

side_by_side.sort_values('faithfulness_gain', ascending=False)



,question,needs_graph,vec_faithfulness,vec_answer_relevancy,vec_total_sec,graph_faithfulness,graph_answer_relevancy,graph_total_sec,faithfulness_gain,latency_delta_sec
1,Which country is the Port of Singapore located in?,True,0.333333,0.000000,0.421274,1.000000,0.972028,0.377807,0.666667,-0.043467
2,Which organization operates in the Port of Rotterdam?,True,0.000000,0.000000,0.411786,0.500000,0.000000,0.404780,0.500000,-0.007006
3,What route does Maersk use according to the graph?,True,0.750000,0.000000,0.576044,1.000000,0.891971,0.386686,0.250000,-0.189358
6,Which port is described as a leading transshipment hub in Southeast Asia?,False,0.750000,0.671960,0.367004,1.000000,0.676159,0.391724,0.250000,0.024719
4,How is Maersk connected to the Panama Canal?,True,0.500000,1.000000,0.547956,0.500000,1.000000,0.370693,0.000000,-0.177263
0,Which port is the largest in Europe according to the lab data?,False,0.500000,0.000000,0.562513,0.333333,0.000000,0.659003,-0.166667,0.096489
5,What do the Panama Canal connect?,False,1.000000,0.421257,0.321165,0.500000,0.995489,0.334051,-0.500000,0.012886
7,What country is Maersk associated with in the lab graph?,False,NaN,0.308005,0.288520,0.500000,0.000000,0.361626,NaN,0.073106


### Step 6e — Save comparison artifacts



| File | Audience |

|------|----------|

| `comparison_summary.csv` | Stakeholders — one row per pipeline |

| `comparison_per_question.csv` | Engineers — drill-down per question |



Archive these files when you change `EVAL_RETRIEVAL_K`, prompts, or models so you can trace regressions.



In [47]:
# Step 6e — Save comparison artifacts

comparison.to_csv(OUTPUT_DIR / 'comparison_summary.csv', index=False)

side_by_side.to_csv(OUTPUT_DIR / 'comparison_per_question.csv', index=False)

print('Saved comparison CSVs')



Saved comparison CSVs


---



# Step 7 — Inspect low-score samples



Aggregate metrics hide **failure modes**. Before changing `k` or prompts, inspect individual rows.



### Debugging workflow



1. **Sort by low faithfulness** — find unsupported claims (hallucinations).

2. **Read retrieved `contexts`** — was evidence missing or ignored?

3. **For GraphRAG** — is `[Graph facts]` empty? Check `MENTIONS` edges in Neo4j Browser.

4. **Compare vector vs graph** on the same question in `comparison_per_question.csv`.



### Common failure patterns



| Pattern | Likely cause |

|---------|--------------|

| Low faithfulness, good retrieval | Model ignored context — tighten prompt or use larger model |

| Low faithfulness, poor retrieval | Wrong chunks — tune `k` or embeddings |

| Empty graph context | Missing `(Chunk)-[:MENTIONS]->(Entity)` links |

| Low relevancy, high faithfulness | Answer too narrow or off-question — prompt or ground-truth wording |



### Step 7a — Lowest faithfulness rows (GraphRAG)



We sort GraphRAG results by **faithfulness ascending**, then **answer relevancy ascending**.



These rows are your **priority fix list** — hallucination risk is highest when faithfulness is near 0.



In [48]:
# Step 7a — Lowest faithfulness rows (GraphRAG)

inspect_cols = ['question', 'answer', 'ground_truth', 'faithfulness', 'answer_relevancy', 'needs_graph']

print('GraphRAG — lowest faithfulness:')

scored_graphrag.sort_values(['faithfulness', 'answer_relevancy'])[inspect_cols].head(3)



GraphRAG — lowest faithfulness:


,question,answer,ground_truth,faithfulness,answer_relevancy,needs_graph
0,Which port is the largest in Europe according to the lab data?,I do not have enough information from the provided context to determine which port is the largest in Europe accordin...,The Port of Rotterdam is the largest port in Europe.,0.333333,0.0,False
2,Which organization operates in the Port of Rotterdam?,I do not have enough information from the provided context to determine which organization operates in the Port of R...,Maersk operates in the Port of Rotterdam.,0.500000,0.0,True
7,What country is Maersk associated with in the lab graph?,I do not have enough information from the provided context.,Maersk is a Danish shipping company (Denmark).,0.500000,0.0,False


### Step 7b — Drill into one row



Change **`row_idx`** to inspect different failures.



For each row, ask:



1. Does **`ground_truth`** match what is actually in the graph/chunks?

2. Does **`answer`** introduce facts not in **`contexts`**?

3. Would vector RAG have scored better? (check `comparison_per_question.csv`)



Use this cell as a template for manual QA before promoting pipeline changes.



In [49]:
# Step 7b — Drill into one row (edit index as needed)

row_idx = 0

row = scored_graphrag.sort_values(['faithfulness', 'answer_relevancy']).iloc[row_idx]

print('Question:', row['question'])

print('\nGraphRAG answer:\n', row['answer'])

print('\nGround truth:\n', row['ground_truth'])

print('\nFaithfulness:', row['faithfulness'], '| Answer relevancy:', row['answer_relevancy'])

print('\nRetrieved contexts:')

for i, ctx in enumerate(row['contexts'], 1):

    print(f'--- context {i} ---\n', ctx[:500])



Question: Which port is the largest in Europe according to the lab data?

GraphRAG answer:
 I do not have enough information from the provided context to determine which port is the largest in Europe according to the lab data. The text only mentions that the Port of Rotterdam is a major hub, but it does not confirm if it is the largest in Europe.

Ground truth:
 The Port of Rotterdam is the largest port in Europe.

Faithfulness: 0.3333333333333333 | Answer relevancy: 0.0

Retrieved contexts:
--- context 1 ---
 The Port of Rotterdam is the largest port in Europe and a major hub for container shipping.
--- context 2 ---
 The Port of Singapore is a leading transshipment hub in Southeast Asia.
--- context 3 ---
 Maersk is a Danish shipping company that operates vessels and uses routes such as the Panama Canal.
--- context 4 ---
 [Graph facts]
Chunk chunk_rotterdam mentions Port_of_Rotterdam; LOCATED_IN -> Netherlands
Chunk chunk_rotterdam mentions Port_of_Rotterdam; OPERATES_IN -> Maersk
C

### Step 7c — Iteration ideas



| Knob | Effect |

|------|--------|

| Increase `EVAL_RETRIEVAL_K` | More chunk context; slower prompts |

| Add corpus chunks (LangChain notebook 3.2.4) | Broader coverage |

| Fix missing `MENTIONS` edges | GraphRAG expansion returns empty |

| Use a larger Ollama model (`llama3.1:8b`) | Better grounding and relevancy |

| Tighten prompts | Reduce unsupported claims |

| Separate labeler model for new ground truth | Avoid circular bias (see Module 4 lab) |



**Workflow:** Change one knob → re-run Steps 4–6 → compare new `comparison_summary.csv` to the previous run.



**Neo4j Browser check for graph issues:**



```cypher

MATCH (c:Chunk:LangChainLab)-[:MENTIONS]->(e)

RETURN c.id, e.id LIMIT 20;

```



---



# Step 8 — Save artifacts and checklist



### Files written to `outputs/eval_graphrag/`



| File | Contents |

|------|----------|

| `ground_truth_graphrag_lab.csv` | Evaluation questions and reference answers |

| `graphrag_eval_raw_outputs.csv` | Answers, contexts, latency, tokens (both pipelines) |

| `scored_vector_rag.csv` | Vector RAG + RAGAS scores |

| `scored_graphrag.csv` | GraphRAG + RAGAS scores |

| `comparison_summary.csv` | Aggregated accuracy / latency / tokens |

| `comparison_per_question.csv` | Side-by-side per question |



### Practical checklist for learners



1. Verify Neo4j lab graph and vector index **before** evaluating (Step 1).

2. Start with a **small**, high-quality ground-truth set (8–15 rows).

3. Compare **vector RAG vs GraphRAG** on `needs_graph=True` questions first.

4. Track **latency and tokens**, not only RAGAS scores.

5. Inspect **low faithfulness** rows before tuning generation (Step 7).

6. Re-run evaluation after each pipeline change and **keep CSV artifacts** for audit.



### What you built



You evaluated a **Neo4j GraphRAG** application with **RAGAS**, compared it to **vector-only RAG**, and measured the **time / cost / accuracy** trade-off — the same workflow teams use before shipping graph-augmented retrieval to production.



### Step 8 — Final summary



The next cell prints the **`comparison`** table — your single-page executive summary.



Look for:



- Which pipeline has higher **faithfulness_mean** and **answer_relevancy_mean**

- Whether GraphRAG's **latency_ratio_vs_vector** is acceptable for your use case

- **faithfulness_mean_needs_graph** — the key metric if graph traversal is your value proposition



In [50]:
# Step 8 — Final summary printout

print('=== Evaluation complete ===')

print(comparison.to_string(index=False))

print('\nArtifacts directory:', OUTPUT_DIR)



=== Evaluation complete ===
  pipeline  faithfulness_mean  answer_relevancy_mean  faithfulness_mean_needs_graph  answer_relevancy_mean_needs_graph  mean_total_sec  p95_total_sec  mean_retrieval_sec  mean_generation_sec  mean_eval_count  total_eval_count  faithfulness_delta_vs_vector  latency_ratio_vs_vector
vector_rag           0.547619               0.300153                       0.395833                              0.250        0.437033       0.571308            0.009080             0.427953            24.75               198                      0.000000                 1.000000
  graphrag           0.666667               0.566956                       0.750000                              0.716        0.410796       0.570025            0.012311             0.398485            20.00               160                      0.119048                 0.939966

Artifacts directory: /home/ethan/newgen/KMOU_Course/Module_8/outputs/eval_graphrag
